## 课程二：多音字语音合成（30%）

实验目标：学习 MeloTTS 的语音前端处理流程

1、文本正则化（Text Normalization，TN）

在语音合成前端中，文本正则化（Text Normalization，TN）是一个预处理步骤，其核心任务是将原始文本中那些无法直接发音的符号、数字或缩写，自动转换为系统能够处理并正确发音的口语化文字形式。

文本正则化处理的非标准词（NSW）范围很广，以下是一些常见类型的转换示例：

| 文本类别 | 原始文本 | 正则化后的口语形式 | 备注 |
| :--- | :--- | :--- | :--- |
| **数字** | `123` | “一百二十三” | 需根据上下文决定是读作基数词还是序数词等。 |
| **百分比** | `20%` | “百分之二十” | 将符号和数字结合转换成完整读法。 |
| **分数** | `1/2` | “二分之一” | 将数学符号转换为对应的口语词汇。 |
| **标点** | `!` | （转换为强烈语气或停顿） | 本身不发音，但会引导后续的韵律处理。 |
| **日期/时间** | `2026-04-07` | “二零二六年四月七日” | 将常见的日期格式转换为自然的口语表达。 |
| **缩写** | `WHO` | “世界卫生组织” | 将首字母缩写展开为全称。 |
| **特殊符号** | `@` | “艾特”或“at” | 在特定语境下（如电子邮件地址）转换为特定读法。 |

MeloTTS 使用了一个名为 `text` 的模块来处理非标准的文本输入，这个模块包含了多个子模块，负责不同的语种的文本处理任务。

>【问题一】请阅读 `melo/text` 下的 `chinese.py`、`english.py`、`chinese_mix.py` 三个脚本，介绍一下 MeloTTS 是如何分别对中文、英文和中英混合文本进行正则化预处理的？  

2、字音转换（Grapheme-to-Phoneme，G2P）

字音转换（Grapheme-to-Phoneme，G2P）是语音合成前端处理中的一个核心模块，其任务是将文本中的每个字符（如字母、汉字）映射为对应的发音符号序列（如音素、拼音），从而为后续的声学模型提供明确的发音指令。

In [ ]:
from melo.text.chinese_mix import text_normalize, g2p, get_bert_feature


text = '我们现在 also 能够 help 很多公司， use some machine learning 的 algorithms 啊!'

# 文本正则化
text = text_normalize(text)
print(f'normalized text: {text}')
print()

# 字音转换
phones, tones, word2ph = g2p(text, impl='v2')
bert = get_bert_feature(text, word2ph, device='cuda:0')
print(f'phones: {phones}')
print(f'tones: {tones}')
print(f'word2ph: {word2ph}')
print()

print(f'phones length: {len(phones)}')
print(f'tones length: {len(tones)}')
print(f'word2ph length: {len(word2ph)}')
print(f'bert.shape: {bert.shape}')

- 这里的 phones （音素）是指语音学中最小的发音单位，中文的音素通常以拼音的形式表示，而英文的音素则使用国际音标（IPA）或其他音标系统来表示。

- tones（声调） 参考以下建模方式：

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| 标点/句边界 | 中文阴平 a | 中文阳平 a | 中文上声 a | 中文去声 a | 中文轻声 a | 日文 | 英文辅音 | 英文元音无重音 | 英文元音主要重音 | 英文元音第二重音 |

这样可以通过 tones 的值区分中英文语种（中文 1~5、英文 7~10）

- word2ph 表示文本单元到音素序列的对齐关系，即每个文本单元对应多少个音素位置，用于将字/词级表示展开到音素级。

下面的代码将更清晰地展示 MeloTTS 中字音转换输出的联合词表的对应关系：

In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)


# 以字为单位分割 text, 中文一个字一个元素，英文单词一个元素
words = []
words.append('_') # 添加起始符
for sub_text in text.split():
    if sub_text.isascii(): # 如果是英文单词，直接添加到列表中
        words.append(sub_text)
    else: # 如果是中文，逐字添加到列表中
        words.extend(list(sub_text))
        
words.append('_') # 添加结束符
print(f'words: {words}')
print(f'word2ph: {word2ph}')

words_repeated = []
repeat_counts = []
for word, repeat in zip(words, word2ph):
    words_repeated.extend([word] * repeat)
    repeat_counts.extend([repeat] * repeat)

df = pd.DataFrame({
    'words': words_repeated,
    'phones': phones,
    'tones': tones,
    'repeat_counts': repeat_counts
})


df.T

words: ['_', '我', '们', '现', '在', 'also', '能', '够', 'help', '很', '多', '公', '司', ',', 'use', 'some', 'machine', 'learning', '的', 'algorithms', '啊', '!', '_']
word2ph: [1, 2, 2, 2, 2, 4, 2, 2, 4, 2, 2, 2, 2, 1, 3, 3, 5, 5, 2, 9, 2, 1, 1]


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60
words,_,我,我,们,们,现,现,在,在,also,also,also,also,能,能,够,够,help,help,help,help,很,很,多,多,公,公,司,司,",",use,use,use,some,some,some,machine,machine,machine,machine,machine,learning,learning,learning,learning,learning,的,的,algorithms,algorithms,algorithms,algorithms,algorithms,algorithms,algorithms,algorithms,algorithms,啊,啊,!,_
phones,_,w,o,m,en,x,ian,z,ai,ao,l,s,ow,n,eng,g,ou,hh,eh,l,p,h,en,d,uo,g,ong,s,i0,",",y,uw,s,s,ah,m,m,ah,sh,iy,n,l,er,n,ih,ng,d,e,ae,l,g,er,ih,dh,ah,m,z,AA,a,!,_
tones,0,3,3,5,5,4,4,4,4,9,7,7,8,2,2,4,4,7,9,7,7,3,3,1,1,1,1,1,1,0,7,9,7,7,9,7,7,8,7,9,7,7,9,7,8,7,5,5,9,7,7,8,10,7,8,7,7,5,5,0,0
repeat_counts,1,2,2,2,2,2,2,2,2,4,4,4,4,2,2,2,2,4,4,4,4,2,2,2,2,2,2,2,2,1,3,3,3,3,3,3,5,5,5,5,5,5,5,5,5,5,2,2,9,9,9,9,9,9,9,9,9,2,2,1,1


3、多音字语音合成

多音字是指在不同语境下具有不同发音的汉字。例如，“行” 在 “银行” 中读作 “háng”，而在 “行走” 中读作 “xíng”。在语音合成中，正确处理多音字是非常重要的，错误的发音会导致合成语音的自然度和可理解性大大降低。

然而，基于规则和上下文分析的方法的文本前端仍然存在一些局限性，不能覆盖所有的多音字情况，特别是在一些复杂的语境中。

例如，山海关孟姜女庙有一副对联：“海水朝朝朝朝朝朝朝落，浮云长长长长长长长消。” 

这副对联百年来难倒了无数文人，后来我国著名文学家、历史学家郭沫若先生在看到这副奇联后，产生了浓厚的兴趣，沉思片刻后，当场就给大家进行了解读，他说：这副对联有很多解读方法，关键之处就在于断句，还要注意 “朝和长” 两个多音字在不同位置的读音，断句不同，读音不同，意思就不一样。

他还给大家举了例子：“海水朝（潮），朝朝朝（潮），朝朝（潮）朝落；浮云长（涨），长长长（涨），长长（涨）长消”

（hǎi shuǐ cháo，zhāo zhāo cháo，zhāo cháo zhāo luò；fú yún zhǎng，cháng cháng zhǎng，cháng zhǎng cháng xiāo。）

海水潮起潮落，浮云聚了又散，人生又何尝不是如此呢！

MeloTTS 不能正确地读出这副对联，会将 “朝” 都读作 “cháo”，将 “长” 都读作 “cháng”，导致合成的语音听起来非常奇怪。

In [ ]:
from melo.api import TTS

# Speed is adjustable
speed = 0.75
device = 'cpu' # or cuda:0

text = "海水朝朝朝朝朝朝朝落，浮云长长长长长长长消。"
model = TTS(language='ZH', device=device)
speaker_ids = model.hps.data.spk2id

output_path = 'zh.wav'

model.tts_to_file(text, speaker_ids['ZH'], output_path, speed=speed)

In [2]:
# 播放生成的音频
from IPython.display import Audio

Audio(output_path)

>【问题二】请根据郭沫若先生的这一种读法，自定义 phones、tones、word2ph 列表，来正确地合成这副对联的语音。

In [3]:
from melo.api import TTS

# Speed is adjustable
speed = 0.75
device = 'cpu' # or cuda:0

# 补充对联中的断句
text = "海水朝，朝朝朝，朝朝朝落；浮云长，长长长，长长长消。"
model = TTS(language='ZH', device=device)
speaker_ids = model.hps.data.spk2id

output_path = 'homework_2.wav'

split_sentences = TTS.split_sentences_into_pieces(text, language='ZH')

print(f'被切分成的句子列表: {split_sentences}，长度: {len(split_sentences)}')

 > Text split to sentences.
海水朝, 朝朝朝, 朝朝朝落.
浮云长, 长长长, 长长长消.
 > ===========================
被切分成的句子列表: ['海水朝, 朝朝朝, 朝朝朝落.', '浮云长, 长长长, 长长长消.']，长度: 2


> hǎi shuǐ cháo，zhāo zhāo cháo，zhāo cháo zhāo luò；fú yún zhǎng，cháng cháng zhǎng，cháng zhǎng cháng xiāo。  
请在下面的代码块中，定义 phones、tones、word2ph 列表，并使用自定义的 `model.tts_to_file_custom_frontend()` 方法来合成这副对联的语音。  
注意事项:  
1、 以子句为单位断句，每个子句的首尾都要添加一个特殊的停顿符号 `_`，以便模型能够更鲁棒地合成语音；  
2、 在对联中人工加入标点符号（如逗号、分号）来更好地控制韵律。  
3、 提示：可以先用正常前端跑出原始 phones, tones, word2ph，再只修改多音字相关位置。

In [ ]:
# TODO: 完成两个子句的 phones、tones、word2ph 列表
phones1 = []
tones1 = []
word2ph1 = []

phones2 = []
tones2 = []
word2ph2 = []

In [ ]:
# 合并两个子句的 phones、tones、word2ph 列表
phones_customized = [phones1, phones2]
tones_customized = [tones1, tones2]
word2ph_customized = [word2ph1, word2ph2]

# 使用自定义的 phones、tones、word2ph 列表进行 TTS 合成
model.tts_to_file_custom_frontend(text, speaker_ids['ZH'], output_path, speed=speed, phones_customized=phones_customized, tones_customized=tones_customized, word2ph_customized=word2ph_customized)

In [ ]:
# 播放生成的音频
from IPython.display import Audio

Audio(output_path)